# JAX Performance Tutorial

A short benchmark notebook comparing the NumPy and JAX execution paths.
It keeps the input small enough to execute quickly, but still exercises the
same EVPI computation on both backends when JAX is installed.

In [1]:
import time

import numpy as np

from voiage.analysis import DecisionAnalysis
from voiage.main_backends import JAX_AVAILABLE
from voiage.schema import ValueArray

rng = np.random.default_rng(11)
strategy_names = ["Current stack", "JAX-accelerated stack", "Hybrid stack"]
nb_values = rng.normal(loc=[120.0, 123.0, 126.0], scale=[8.0, 7.0, 9.0], size=(4000, 3))
value_array = ValueArray.from_numpy(nb_values, strategy_names)

numpy_analysis = DecisionAnalysis(value_array, backend="numpy")
numpy_evpi = numpy_analysis.evpi()
numpy_timings = []
for _ in range(5):
    start = time.perf_counter()
    _ = numpy_analysis.evpi()
    numpy_timings.append(time.perf_counter() - start)

print(f"NumPy EVPI: {numpy_evpi:.4f}")
print(f"NumPy median runtime: {np.median(numpy_timings):.6f}s")

if JAX_AVAILABLE:
    jax_analysis = DecisionAnalysis(value_array, backend="jax", use_jit=True)
    _ = jax_analysis.evpi()  # warm-up / compile
    jax_timings = []
    for _ in range(5):
        start = time.perf_counter()
        _ = jax_analysis.evpi()
        jax_timings.append(time.perf_counter() - start)

    jax_evpi = jax_analysis.evpi()
    print(f"JAX EVPI: {jax_evpi:.4f}")
    print(f"JAX median runtime: {np.median(jax_timings):.6f}s")
    print(f"Results agree: {np.isclose(numpy_evpi, jax_evpi, rtol=1e-6, atol=1e-6)}")
else:
    print("JAX is not installed in this environment, so only the NumPy path was benchmarked.")

NumPy EVPI: 4.4118
NumPy median runtime: 0.000099s


JAX EVPI: 4.4118
JAX median runtime: 0.028136s
Results agree: True


In [2]:
if JAX_AVAILABLE:
    from voiage.backends.performance_profiler import JaxPerformanceProfiler

    profiler = JaxPerformanceProfiler()
    comparison = profiler.compare_implementations(
        numpy_analysis.evpi,
        jax_analysis.evpi,
        (),
        n_runs=3,
    )
    print("Comparison summary:")
    print(f"  mean NumPy time: {np.mean(comparison['numpy_times']):.6f}s")
    print(f"  mean JAX time:   {np.mean(comparison['jax_times']):.6f}s")
    print(f"  mean speedup:    {np.mean(comparison['speedups']):.3f}x")
else:
    print("Skipping profiler comparison because JAX is unavailable.")

Comparison summary:
  mean NumPy time: 0.000194s
  mean JAX time:   0.026696s
  mean speedup:    0.007x
